**프로모션별 판매량 예측 머신러닝 학습 및 사용 예시 코드**

- 전처리에서 최종적으로 만든 데이터 사용 및 사용할 대분류 정의 필요

- 경로 수정 필요

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.ensemble import RandomForestRegressor

## 데이터 불러오기

In [ ]:
# 불러올 카테고리 리스트
lr_list = ['유음료','과자']

# promotion_df 저장 리스트
promotion_df_list = []

for LR_NM in lr_list:
    # 파일 경로 설정
    file_path = f'{LR_NM}/{LR_NM}_프로모션.csv'
    
    # CSV 읽기
    temp_df = pd.read_csv(file_path)    
    promotion_df_list.append(temp_df)

# 유음료 + 빵 데이터 통합
promotion_df = pd.concat(promotion_df_list, ignore_index=True)

# 확인
# print(promotion_df.head())
print(promotion_df.shape)

# sale_df들을 저장할 리스트
sale_df_list = []

for LR_NM in lr_list:
    # 파일 경로 설정
    file_path = f'{LR_NM}/{LR_NM}_판매수량_최종_스테디.csv'
    
    # CSV 파일 읽기
    temp_df = pd.read_csv(file_path)
    sale_df_list.append(temp_df)

# 두 카테고리 데이터를 하나의 DataFrame으로 통합
sale_df = pd.concat(sale_df_list, ignore_index=True)

# 최종 확인
# print(sale_df.head())
print(sale_df.shape)

In [ ]:
sale_cut = sale_df[['기준일자','점포코드','상품코드','상품중분류코드', '상품소분류코드','판매수량_최종','매출매가금액']]
promo_cut = promotion_df[['기준일자','점포코드','점포명','상품코드','상품명','MM구분','행사명','행사코드','판매수량','행사판매금액','정상판매금액','행사시작일자','행사종료일자','영수증번호','원가']]

## 판매 데이터 전처리
sale_cut.loc[:, '기준일자'] = pd.to_datetime(sale_cut['기준일자'], errors='coerce').dt.date
sale_cut['기준일자'] = pd.to_datetime(sale_cut['기준일자'])

sale_cut.loc[:, '상품코드'] = sale_cut['상품코드'].astype(str)
sale_cut.loc[:, '상품중분류코드'] = sale_cut['상품중분류코드'].astype(str)
sale_cut.loc[:, '상품소분류코드'] = sale_cut['상품소분류코드'].astype(str)
sale_cut.loc[:, '점포코드'] = sale_cut['점포코드'].astype(str)

cols_to_convert = ['판매수량_최종','매출매가금액']

sale_cut.loc[:, cols_to_convert] = (
    sale_cut[cols_to_convert]
    .replace(',', '', regex=True)
    .astype(float)
)

## 프로모션 데이터 전처리
promo_cut = promo_cut[promo_cut['MM구분']!='콤보증정']

promo_cut.loc[:, '기준일자'] = pd.to_datetime(promo_cut['기준일자'], errors='coerce').dt.date
promo_cut.loc[:, '행사시작일자'] = pd.to_datetime(promo_cut['행사시작일자'], errors='coerce').dt.date
promo_cut.loc[:, '행사종료일자'] = pd.to_datetime(promo_cut['행사종료일자'], errors='coerce').dt.date
promo_cut['기준일자'] = pd.to_datetime(promo_cut['기준일자'])

promo_cut.loc[:, '상품코드'] = promo_cut['상품코드'].astype(str)
promo_cut.loc[:, '영수증번호'] = promo_cut['영수증번호'].astype(str)
promo_cut.loc[:, '점포코드'] = promo_cut['점포코드'].astype(str)

cols_to_convert = ['판매수량','행사판매금액','정상판매금액','원가']

promo_cut.loc[:, cols_to_convert] = (
    promo_cut[cols_to_convert]
    .replace(',', '', regex=True)
    .astype(float)
)

## 전처리

In [ ]:
def define_continuous_action_discount_rate(promotion_df):
    """
    프로모션 할인율(discount_rate) 계산 함수
    - 기본 할인율: 행사판매금액 / (정상판매금액 * 판매수량) -> 행사판매금액 대신에 정상판매금액*판매수량 - 판매금액으로 해야할수도
    - BOGO(1+1, 2+1 등) 형태의 프로모션이면 BOGO 할인율로 덮어쓰기

    BOGO 할인율 = free_unit / (buy_unit + free_unit)
    예: 1+1 → 1 / (1+1) = 0.5
        2+1 → 1 / (2+1) = 0.3333
    """

    # ------------------------
    # 1) 기본 할인율 계산
    # ------------------------
    promotion_sale = promotion_df[promotion_df['판매수량']>=0].copy()
    promotion_sale['discount_rate'] = (
        promotion_sale['행사판매금액'] /
        (promotion_sale['정상판매금액'] * promotion_sale['판매수량'])
    )

    # ------------------------
    # 2) BOGO 패턴 체크 + 덮어쓰기
    # ------------------------
    def calc_bogo_rate(row):
        name = str(row['행사명'])

        # BOGO 패턴: (\d+)\s*\+\s*(\d+)
        match = re.search(r'(\d+)\s*\+\s*(\d+)', name)

        if match:
            buy = int(match.group(1))
            free = int(match.group(2))
            total = buy + free
            # free unit 비율 = free / total
            return free / total

        # 패턴 없으면 기존 discount_rate 유지
        return row['discount_rate']

    promotion_sale['discount_rate'] = promotion_sale.apply(calc_bogo_rate, axis=1)

    return promotion_sale

In [ ]:
# 데이터에 할인율 추가
promo_cut2 = promo_cut[['기준일자','점포코드','상품코드','판매수량','MM구분','행사명','행사코드','행사판매금액','정상판매금액','행사시작일자','행사종료일자']].copy()
promo_dc_rate = define_continuous_action_discount_rate(promo_cut2)

# 번들증정 행사인데 + 없는 케이스 제거
cond_remove = (
    ~(promo_dc_rate['행사명'].str.contains('+', regex=False)) &
    (promo_dc_rate['MM구분'] == '번들증정')
)

# 조건에 해당하는 행 삭제
promo_dc_rate_filtered = promo_dc_rate[~cond_remove].copy()

promo_dc_rate = promo_dc_rate_filtered

# 기준일자, 점포, 상품별로 그룹화
promo_group = promo_dc_rate.groupby(['기준일자','점포코드','상품코드','행사코드'])['discount_rate'].unique().reset_index()

# discount_rate가 한개만 들어간 행만 필터링
promo_group_clean = promo_group[promo_group['discount_rate'].apply(lambda x: len(x) == 1)].copy()
print('프로모션 그룹화',promo_group_clean.shape)
promo_group_clean.head()

In [ ]:
# 빈 날짜 데이터 생성
date_range = pd.date_range(start="2023-01-01", end="2024-12-31")
timestamp = pd.DataFrame({'기준일자': date_range})
timestamp['기준일자'] = pd.to_datetime(timestamp['기준일자'])

# discount_rate값을 array에서 단일값으로 변환
promo_group_clean['discount_rate'] = promo_group_clean['discount_rate'].apply(lambda x: x[0]) 

# 병합
tmp = pd.merge(timestamp, sale_cut, on=['기준일자'], how='left')
result = pd.merge(tmp, promo_group_clean, on=['기준일자','상품코드','점포코드'], how='left')

result = result.sort_values(['기준일자','점포코드','상품코드'])

# 요일 (0=월요일, 6=일요일)
result['weekday'] = result['기준일자'].dt.weekday

# 연중 주차 (ISO week number)
result['week_of_year'] = result['기준일자'].dt.isocalendar().week.astype(int)

print(result.shape)
result.head()

In [ ]:
result = sale_cut.merge(promo_group_clean, 
                        on=['기준일자','상품코드','점포코드'], how='left')

result['weekday'] = result['기준일자'].dt.weekday
result['week_of_year'] = result['기준일자'].dt.isocalendar().week.astype(int)

print(result.shape)
result.head()

### 할인율이 비어있는 행에 대해 할인율 추가

In [ ]:
# 프로모션 데이터에서 행사별 진행 날짜 추출
promo_item_list = (
    promotion_df.groupby(['행사코드','행사명','행사시작일자','행사종료일자'])
      .agg(
          점포코드=('점포코드', lambda x: sorted(x.unique().tolist())),
          점포목록=('점포명', lambda x: sorted(x.unique().tolist())),
          상품코드목록=('상품코드', lambda x: sorted(x.unique().tolist())),
          상품명목록=('상품명',   lambda x: sorted(x.unique().tolist()))
      )
      .reset_index()
)

for col in ['행사시작일자', '행사종료일자']:
    promo_item_list[col] = promo_item_list[col].astype(str)
    promo_item_list.loc[promo_item_list[col] == '99991231', col] = '20251231'


promo_item_list.tail()

In [ ]:
# 데이터에서 할인율이 비어 있는 행에 대해, 점포·상품·일자 기준으로 유효한 행사코드 삽입

# 0. 날짜 변환
result['기준일자'] = pd.to_datetime(result['기준일자'])
promo_item_list['행사시작일자'] = pd.to_datetime(promo_item_list['행사시작일자'])
promo_item_list['행사종료일자'] = pd.to_datetime(promo_item_list['행사종료일자'])

# 1. promo_item_list의 점포·상품 목록(list)을 펼쳐서 join 가능한 구조 생성
promo_expanded = (
    promo_item_list
        .explode('점포코드')       # 점포코드 리스트 해체
        .explode('상품코드목록')  # 상품코드 리스트 해체
        [['행사코드','행사명','행사시작일자','행사종료일자',
          '점포코드','상품코드목록']]
        .rename(columns={'상품코드목록':'상품코드'})
)

# 타입 통일
result['점포코드'] = result['점포코드'].astype(str)
result['상품코드'] = result['상품코드'].astype(str)
promo_expanded['점포코드'] = promo_expanded['점포코드'].astype(str)
promo_expanded['상품코드'] = promo_expanded['상품코드'].astype(str)

# 2. 할인율 NaN 행만 추출
mask_nan = result['discount_rate'].isna()
target = result[mask_nan][['기준일자','점포코드','상품코드']]

# 3. 점포코드 + 상품코드 기준으로 행사 후보 매칭 (행 단위 apply 제거)
merged = target.merge(
    promo_expanded,
    on=['점포코드','상품코드'],
    how='left'
)

# 4. 행사기간 조건 필터(벡터 연산)
mask_date = (
    (merged['기준일자'] >= merged['행사시작일자']) &
    (merged['기준일자'] <= merged['행사종료일자'])
)

merged_valid = merged[mask_date]

# 5. 기준일자+점포코드+상품코드 조합당 행사코드 1개 매칭
picked_event = (
    merged_valid
    .groupby(['기준일자','점포코드','상품코드'], as_index=False)
    .agg({'행사코드': 'first'})
)

# 6. result의 행사코드 채우기
result_test = result.copy()

result_test = result_test.merge(
    picked_event,
    on=['기준일자','점포코드','상품코드'],
    how='left',
    suffixes=('', '_new')
)

# NaN인 행사코드만 새 값으로 채우기
result_test['행사코드'] = result_test['행사코드'].fillna(result_test['행사코드_new'])

# 보조 컬럼 제거
result_test.drop(columns=['행사코드_new'], inplace=True)

In [ ]:
# 1. 행사코드–상품코드별 discount_rate 매핑 테이블 생성
discount_map = (
    promo_group_clean[['행사코드','상품코드','discount_rate']]
    .dropna(subset=['discount_rate'])
    .drop_duplicates(subset=['행사코드','상품코드'])
)

# 2. 수정 대상 행 추출
mask = (result_test['행사코드'].notna()) & (result_test['discount_rate'].isna())

target = result_test.loc[mask, ['행사코드','상품코드']].copy()

# 3. discount_rate lookup 수행
target = target.merge(
    discount_map,
    on=['행사코드','상품코드'],
    how='left'
)

# 4. result_test discount_rate 업데이트
result_test.loc[mask, 'discount_rate'] = target['discount_rate'].values


In [ ]:
result_test['discount_rate'] = result_test['discount_rate'].fillna(0)
result_test['promo_flag'] = (result_test['discount_rate'] > 0).astype(int)

df = result_test[['기준일자','점포코드','상품코드','상품중분류코드','상품소분류코드','판매수량_최종','매출매가금액','discount_rate','promo_flag','weekday','week_of_year']].copy()
print(df.shape)
df.head()

### 필터링 조건 생성

In [ ]:
# 전체 판매일이 6개월 미만인 점포 
store_days = sale_df.groupby('점포코드')['기준일자'].nunique().sort_values().reset_index()
recent_store_code = store_days[store_days['기준일자']<180]['점포코드'].tolist()

promotion_df_cut = promotion_df[~promotion_df['점포코드'].isin(recent_store_code)]
sale_df_cut = sale_df[~sale_df['점포코드'].isin(recent_store_code)]
print(sale_df_cut['점포코드'].nunique())

In [ ]:
# 프로모션, 비프로모션 일수가 각각 1개월(28일) 이상인 SKU 선별
goods_master = pd.read_csv('(공통)상품마스터_한글컬럼.csv')

# 기준 기간 설정
start_ref = pd.to_datetime("2023-01-01")
end_ref   = pd.to_datetime("2024-12-31")

# 날짜 전처리
promotion_df_cut['행사시작일자'] = pd.to_datetime(promotion_df_cut['행사시작일자'], errors='coerce')
promotion_df_cut['행사종료일자'] = pd.to_datetime(promotion_df_cut['행사종료일자'], errors='coerce')

goods_master["개시일자(출시일자)"] = pd.to_datetime(
    goods_master["개시일자(출시일자)"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

# 1. 점포코드 × 상품코드 × 프로모션기간 계산 함수

def compute_promo_days(df):
    """
    df: 특정 '상품코드 * 점포코드'의 프로모션 기간들
    날짜 병합(Overlapping merge) 후 총 프로모션 일수 반환
    """
    if df.empty:
        return 0

    # 기준 기간으로 클램프
    df["행사시작일자"] = df["행사시작일자"].apply(lambda x: max(x, start_ref))
    df["행사종료일자"] = df["행사종료일자"].apply(lambda x: min(x, end_ref))

    # 유효하지 않은 기간 제거
    df = df[df["행사시작일자"] <= df["행사종료일자"]]
    if df.empty:
        return 0

    df = df.sort_values("행사시작일자").reset_index(drop=True)

    merged = []
    cur_s = df.loc[0, "행사시작일자"]
    cur_e = df.loc[0, "행사종료일자"]

    for i in range(1, len(df)):
        s, e = df.loc[i, "행사시작일자"], df.loc[i, "행사종료일자"]

        if s <= cur_e:
            cur_e = max(cur_e, e)
        else:
            merged.append([cur_s, cur_e])
            cur_s, cur_e = s, e

    merged.append([cur_s, cur_e])

    total_days = sum((e - s).days + 1 for s, e in merged)
    return total_days


# 2. 점포 × 상품 단위로 프로모션 일수 계산
promo_days = (
    promotion_df_cut
    .groupby(["점포코드", "상품코드"])
    .apply(compute_promo_days)
    .reset_index()
)

promo_days.columns = ["점포코드", "상품코드", "프로모션_일수"]


# 3. 전체 가능일수 계산 (상품별 출시일 기준)
def compute_total_days(row):
    opening = row["개시일자(출시일자)"]

    if pd.isna(opening) or opening < start_ref:
        start_date = start_ref
    else:
        start_date = opening

    return (end_ref - start_date).days + 1


sku_open = goods_master[["상품코드", "개시일자(출시일자)"]].drop_duplicates()
sku_open["전체_가능일수"] = sku_open.apply(compute_total_days, axis=1)

# 4. 점포 × 상품 단위로 전체 가능일수 병합
promo_merge_sku_store = promo_days.merge(
    sku_open,
    on="상품코드",
    how="left"
)

promo_merge_sku_store["프로모션_일수"] = promo_merge_sku_store["프로모션_일수"].fillna(0)

promo_merge_sku_store["비프로모션_일수"] = (
    promo_merge_sku_store["전체_가능일수"] - promo_merge_sku_store["프로모션_일수"]
)


# 5. 점포별로 프로모션일수 / 비프로모션일수 1개월 이상인 SKU 선별
result = promo_merge_sku_store[
    (promo_merge_sku_store["프로모션_일수"] >= 28) &
    (promo_merge_sku_store["비프로모션_일수"] >= 28)
].copy()

# 점포별 상품코드 리스트 생성
promo_days_code_by_store = (
    result.groupby("점포코드")["상품코드"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .reset_index()
)

promo_days_code_by_store.head()

In [ ]:
# 0. 점포 목록
store_list = sale_df['점포코드'].unique().tolist()

# 1. 미리 계산 가능한 것들 먼저 계산
# (A) SKU별 총판매수량
qty_sum_all = (
    sale_df.groupby(['점포코드', '상품코드'])['판매수량_최종']
    .sum()
    .rename('qty_sum')
)

# (B) SKU별 판매일수
days_sold_all = (
    sale_df[sale_df["판매수량_최종"] > 0]
    .groupby(['점포코드', '상품코드'])['기준일자']
    .nunique()
    .rename('days_sold')
)

# (C) 프로모션 사용 SKU (점포단위 set)
promo_sku_map = (
    promotion_df.groupby('점포코드')['상품코드']
    .unique()
    .apply(set)
    .to_dict()
)

# (D) 점포별 IQR 사전 계산
Q1 = qty_sum_all.groupby('점포코드').quantile(0.25).rename('Q1')
Q3 = qty_sum_all.groupby('점포코드').quantile(0.75).rename('Q3')
IQR = (Q3 - Q1).rename('IQR')

# (E) 점포별 median (판매량)
median_qty_map = (
    qty_sum_all.reset_index()
    .groupby('점포코드')['qty_sum']
    .median()
    .rename('median_qty')
)

# (F) 점포별 median (판매일수)
median_days_map = (
    days_sold_all.groupby('점포코드')
    .median()
    .rename('median_days')
)

# 최종 결과 저장 딕셔너리
store_sku_filter = {}

# 점포별 Loop 
for store in store_list:

    # 해당 점포의 데이터 slice
    qty_store = qty_sum_all.loc[store] if store in qty_sum_all.index.levels[0] else pd.Series()
    days_store = days_sold_all.loc[store] if store in days_sold_all.index.levels[0] else pd.Series()

    # 1) 프로모션 SKU
    promo_sku = promo_sku_map.get(store, set())

    # 2) 판매수량 median 이상
    qty_threshold  = median_qty_map.get(store, 0) * 0.8

    sku_keep_qty = qty_store[qty_store >= qty_threshold].index.tolist()

    # 3) IQR 이상치 제거
    store_Q1 = Q1.get(store, -999999)
    store_Q3 = Q3.get(store, 999999)
    store_IQR = IQR.get(store, 0)

    lower = store_Q1 - 1.5 * store_IQR
    upper = store_Q3 + 1.5 * store_IQR

    valid_codes = qty_store[(qty_store >= lower) & (qty_store <= upper)].index.tolist()

    # 4) 판매일수 median 이상
    days_threshold = median_days_map.get(store, 0) * 0.8

    sku_keep_days = days_store[days_store >= days_threshold].index.tolist()

    # 5) 최종 교집합
    final_set = (
        set(promo_sku)
        # & set(sku_keep_qty)
        & set(valid_codes)
        # & set(sku_keep_days)
    )

    store_sku_filter[store] = sorted(list(final_set))


In [ ]:
# 각 조건을 통과한 SKU들에 대해, 점포별로 프로모션 일수 기준 SKU 리스트와 교집합 계산
final_intersection = []

for _, row in promo_days_code_by_store.iterrows():
    store = row['점포코드']
    promo_list = row['상품코드']            # 리스트 형식
    sku_set_1 = set(promo_list)

    # store_sku_filter에 점포가 없는 경우 skip
    sku_set_2 = set(store_sku_filter.get(store, []))

    # 교집합
    intersect = sku_set_1 & sku_set_2

    final_intersection.append({
        '점포코드': store,
        '공통_상품코드목록': sorted(list(intersect)),
        '공통_상품수': len(intersect)
    })

final_intersection_df = pd.DataFrame(final_intersection)


In [ ]:
# final_intersection_df 의 리스트 컬럼 explode
tmp = final_intersection_df[['점포코드', '공통_상품코드목록']].explode('공통_상품코드목록')

tmp = tmp.rename(columns={'공통_상품코드목록': '상품코드'})
tmp['상품코드'] = tmp['상품코드'].astype(str)
tmp['점포코드'] = tmp['점포코드'].astype(str)

# df에서도 타입 통일
df['점포코드'] = df['점포코드'].astype(str)
df['상품코드'] = df['상품코드'].astype(str)
# df['상품코드'] = df['상품코드'].astype(str).str.zfill(6)


# merge로 필터링 (inner join)
df_filter = df.merge(tmp, on=['점포코드', '상품코드'], how='inner')

print(df_filter.shape)
df_filter.head()

# 모델 학습

In [ ]:
# 할인율 기반 프로모션 유형 분류 함수

def classify_promo_type(rate):
    # NaN 또는 음수 등 이상값 처리
    if pd.isna(rate) or rate < 0:
        return "unknown"

    # 0 → 무프로모션
    if np.isclose(rate, 0.0, atol=0.01):
        return "non_promo"
    
    # 2+1 → 약 33.33%
    if np.isclose(rate, 1/3, atol=0.01):   # 0.333 ± 0.01
        return "bundle_2plus1"
    
    # 1+1 → 50%
    if np.isclose(rate, 0.5, atol=0.01):
        return "bundle_1plus1"
    
    # 그 외는 퍼센트 할인
    return "percent"

In [ ]:
# 0. 기본 전처리
df = df_filter.copy()
df['기준일자'] = pd.to_datetime(df['기준일자'])

# 카테고리 변환
df['item_md_code'] = pd.to_numeric(df['상품중분류코드'], errors='coerce').astype('Int64').astype('category')
df['item_sm_code'] = pd.to_numeric(df['상품소분류코드'], errors='coerce').astype('Int64').astype('category')
df['store_code']   = pd.to_numeric(df['점포코드'], errors='coerce').astype('Int64').astype('category')
df['item_code']    = pd.to_numeric(df['상품코드'], errors='coerce').astype('Int64').astype('category')

# 프로모션 카테고리 추가
df['promo_type'] = df['discount_rate'].apply(classify_promo_type)
df['promo_type'] = df['promo_type'].astype('category')

# 가격 단일화
df['price'] = df['매출매가금액'] / df['판매수량_최종']
df['price'].replace([np.inf, -np.inf], np.nan, inplace=True)

df = df.sort_values(['기준일자', '상품코드'])
df['price'] = df.groupby('상품코드')['price'].ffill()

def conditional_bfill(s):
    return s.bfill() if pd.isna(s.iloc[0]) else s

df['price'] = df.groupby('상품코드')['price'].transform(conditional_bfill)


# 1. Train/Valid/Test split
train_end   = pd.to_datetime("2024-08-31")
valid_start = pd.to_datetime("2024-09-01")
valid_end   = pd.to_datetime("2024-10-31")
test_start  = pd.to_datetime("2024-11-01")

train_df = df[(df['기준일자'] >= "2023-01-01") & (df['기준일자'] <= train_end)]
valid_df = df[(df['기준일자'] >= valid_start) & (df['기준일자'] <= valid_end)]
test_df  = df[df['기준일자'] >= test_start]

print(len(train_df), len(valid_df), len(test_df))


# 2. item_code × store_code 단위 baseline 생성
group_cols = ['item_code', 'store_code']

baseline_df = (
    train_df
    .groupby(group_cols)
    .agg(
        mean_qty=('판매수량_최종', 'mean'),
        mean_price=('price', 'mean'),
        mean_discount=('discount_rate', 'mean')
    )
    .reset_index()
)

# 3. baseline merge
train_df = train_df.merge(baseline_df, on=group_cols, how='left')
valid_df = valid_df.merge(baseline_df, on=group_cols, how='left')
test_df  = test_df.merge(baseline_df, on=group_cols, how='left')


# 4. delta 계산 
for df_ in [train_df, valid_df, test_df]:
    df_['qty_delta']      = df_['판매수량_최종'] - df_['mean_qty']
    df_['price_delta']    = df_['price'] - df_['mean_price']
    df_['discount_delta'] = df_['discount_rate'] - df_['mean_discount']


# 5. delta_features 구성
delta_features = [
    'discount_delta',
    'price_delta',
    'discount_rate',
    'price',
    'weekday',
    'week_of_year',
    'item_md_code',
    'item_sm_code',
    'store_code',
    'item_code',
    'promo_type'
]

# 6. 카테고리 인코딩
cat_cols = ['item_md_code', 'item_sm_code', 'store_code', 'item_code', 'promo_type']

def encode_cat(df, features):
    X = df[features].copy()
    for c in cat_cols:
        X[c] = X[c].cat.codes.astype(int)
    return X

X_train_enc = encode_cat(train_df, delta_features)
X_valid_enc = encode_cat(valid_df, delta_features)
X_test_enc  = encode_cat(test_df,  delta_features)


In [ ]:
# 최종 random forest 모델 학습 및 예측
rf2 = RandomForestRegressor(
    n_estimators=600,
    max_depth=12,
    min_samples_leaf=250,
    min_samples_split=500,
    max_features=0.6,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)


rf2.fit(X_train_enc, train_df['qty_delta'])

rf_delta_pred2 = rf2.predict(X_test_enc)
rf_qty_pred2 = rf_delta_pred2 + test_df['mean_qty'].values

In [ ]:
# 모델 및 컬럼 정보 저장
sklearn_artifacts = {
    "rf_model_conservative": rf2,
    "X_train_columns": list(X_train_enc.columns)
}

with open(f"randomforest_conservative_delta_artifacts2.pkl", "wb") as f:
    pickle.dump(sklearn_artifacts, f)

# 프로모션 추천 모델 사용 예시 - 경희대 3호점

## 예측용 baseline 데이터 생성

In [ ]:
# 0. 원본 데이터 복사 및 기본 전처리
df = df_filter.copy()

# 날짜 타입 변환
df['기준일자'] = pd.to_datetime(df['기준일자'])

# 가격 계산 (판매수량 0인 경우 inf 방지)
df['price'] = df['매출매가금액'] / df['판매수량_최종']
df['price'].replace([np.inf, -np.inf], np.nan, inplace=True)


# 1. 최근 6주(42일) 데이터만 사용한다고 가정
latest_date = df['기준일자'].max()
start_date = latest_date - pd.Timedelta(days=41)

df_6w = df[
    (df['기준일자'] >= start_date) &
    (df['기준일자'] <= latest_date)
].copy()


# 2. 점포 × 상품 baseline 생성
#    - 최근 6주 내 등장한 상품만 생성됨
#    - 없는 상품은 자동으로 fallback 대상
store_sku_baseline = (
    df_6w
    .groupby(['점포코드', '상품코드'])
    .agg(
        mean_qty=('판매수량_최종', 'mean'),
        mean_price=('price', 'mean'),
        mean_discount=('discount_rate', 'mean')
    )
    .reset_index()
)

# 3. 점포 × 소분류 baseline 생성 - SKU baseline이 없을 경우 사용
store_sm_baseline = (
    df_6w
    .groupby(['점포코드', '상품소분류코드'])
    .agg(
        mean_qty=('판매수량_최종', 'mean'),
        mean_price=('price', 'mean'),
        mean_discount=('discount_rate', 'mean')
    )
    .reset_index()
)

# 4. 경희대3호점만
store_sku_baseline_kh3 = store_sku_baseline[store_sku_baseline['점포코드']=='10734']
store_sm_baseline_kh3 = store_sm_baseline[store_sm_baseline['점포코드']=='10734']

In [ ]:
# 상품 코드 결합 + 소분류, 중분류 추가
item_codes = pd.read_csv('(공통)상품마스터_한글컬럼.csv', encoding='utf-8-sig')
item_codes_only = item_codes[['상품코드','상품중분류코드','상품소분류코드']]

item_codes_only['상품코드'] = item_codes_only['상품코드'].astype(str)
store_sku_baseline_kh3['상품코드'] = (
    store_sku_baseline_kh3['상품코드']
    .astype(str)
    .str.zfill(6)
)

store_sku_baseline_kh3 = store_sku_baseline_kh3.merge(item_codes_only, on='상품코드', how='left')

item_codes_only['상품소분류코드'] = item_codes_only['상품소분류코드'].astype(int)
store_sm_baseline_kh3['상품소분류코드'] = (
    store_sm_baseline_kh3['상품소분류코드']
    .astype(float)
    .astype(int)
)

store_sm_baseline_kh3 = store_sm_baseline_kh3.merge(item_codes_only[['상품소분류코드','상품중분류코드']], on='상품소분류코드', how='left')

In [ ]:
store_sku_baseline_kh3 = store_sku_baseline_kh3.rename(columns={
    '점포코드': 'store_code',
    '상품코드': 'item_code'
})

store_sm_baseline_kh3 = store_sm_baseline_kh3.rename(columns={
    '점포코드': 'store_code',
    '상품코드': 'item_code'
})

In [ ]:
store_sm_baseline_with_item_kh3 = (
    store_sm_baseline_kh3
    .merge(
        item_codes_only[['상품코드', '상품소분류코드']],
        on='상품소분류코드',
        how='left'
    )
    .rename(columns={'상품코드': 'item_code'})
)

store_sm_baseline_with_item_kh3 = store_sm_baseline_with_item_kh3.drop_duplicates()


In [ ]:
store_sku_baseline_kh3.to_csv('경희대3호점_예시데이터\kh3_sku_baseline_str.csv', encoding='utf-8-sig', index=False)
store_sm_baseline_with_item_kh3.to_csv('경희대3호점_예시데이터\kh3_sm_drop_baseline.csv', encoding='utf-8-sig', index=False)

## 프로모션 추천 사용 예시 

### 프로모션 추천 로직 함수

In [ ]:
def make_features(date):
    """요일 및 주차 데이터 생성"""
    dt = pd.to_datetime(date)
    return {
        "weekday": dt.weekday(),
        "week_of_year": int(dt.isocalendar().week)
    }

def classify_promo_type(rate):
    """할인율 기반 프로모션 유형 분류"""
    if np.isclose(rate, 0.0, atol=0.01):
        return "non_promo"
    if np.isclose(rate, 1/3, atol=0.01):
        return "bundle_2plus1"
    if np.isclose(rate, 0.5, atol=0.01):
        return "bundle_1plus1"
    return "percent"

# 원하는 할인율 설정
DISCOUNT_GRID = np.array([0.0, 0.125, 0.33, 0.5])

def get_baseline_row(
    store_code,
    item_code,
    sku_baseline,
    sm_baseline
):
    """
    baseline 조회 함수 
    - SKU(상품코드)가 존재하면 해당 행 반환
    - 없으면 소분류 기준 fallback
    """
    # 1) SKU baseline
    sku_row = sku_baseline[
        (sku_baseline["store_code"] == store_code) &
        (sku_baseline["item_code"] == item_code)
    ]

    if len(sku_row) >= 1:
        return sku_row.iloc[0], "sku"

    # 2) SM baseline fallback (item_code 기준)
    sm_row = sm_baseline[
        (sm_baseline["store_code"] == store_code) &
        (sm_baseline["item_code"] == item_code)
    ]

    if len(sm_row) >= 1:
        return sm_row.iloc[0], "sm"

    return None, "none"


In [ ]:
PROMO_TYPE_MAP = {
    "non_promo": 0,
    "bundle_2plus1": 1,
    "bundle_1plus1": 2,
    "percent": 3,
    "unknown": -1
}

def build_model_input(
    baseline_row,
    store_code,
    item_code,
    weekday,
    week_of_year
):
    """모델 입력 데이터프레임 생성"""
    rows = []

    for d in DISCOUNT_GRID:
        rows.append({
            "discount_rate": d,
            "discount_delta": d - baseline_row["mean_discount"],
            "price": baseline_row["mean_price"],
            "price_delta": 0.0,

            "weekday": weekday,
            "week_of_year": week_of_year,

            "store_code": store_code,
            "item_code": item_code,
            "item_md_code": baseline_row["상품중분류코드"],
            "item_sm_code": baseline_row["상품소분류코드"],

            "promo_type": PROMO_TYPE_MAP[classify_promo_type(d)]

        })

    return pd.DataFrame(rows)

def predict_demand_curve_rf(
    model,
    X,
    mean_qty,
    X_train_columns
):
    """
    랜덤포레스트 모델을 이용해 평균 수요 대비 증감량을 예측하고 최종 수요 예측값을 계산
    """
    X_model = align_rf_input(X, X_train_columns)

    delta_pred = model.predict(X_model)
    qty_pred = np.maximum(mean_qty + delta_pred, 0)

    result = X.copy()
    result["pred_qty"] = qty_pred
    return result


def align_rf_input(X, X_train_columns):
    """예측용 데이터의 컬럼을 학습 시 사용한 컬럼 구조에 맞게 정렬하고 누락된 컬럼은 0으로 생성"""
    X_aligned = pd.DataFrame(
        0.0,
        index=X.index,
        columns=X_train_columns
    )

    for c in X.columns:
        if c in X_aligned.columns:
            X_aligned[c] = X[c].values

    return X_aligned

def predict_cumulative_curve_rf(
    model,
    baseline_row,
    store_code,
    item_code,
    start_date,
    end_date,
    X_train_columns
):
    """
    지정한 기간(start_date~end_date) 동안 날짜별 feature를 반영해 할인율별 일판매량을 예측
    이를 합산한 누적 판매량(cum_pred_qty) 곡선을 생성
    """
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)

    if end_dt < start_dt:
        raise ValueError("end_date must be >= start_date")

    dates = pd.date_range(start_dt, end_dt, freq="D")

    X_list = []
    for cur_dt in dates:
        feat = make_features(cur_dt)
        X_day = build_model_input(
            baseline_row=baseline_row,
            store_code=store_code,
            item_code=item_code,
            weekday=feat["weekday"],
            week_of_year=feat["week_of_year"]
        )
        X_day = X_day.copy()
        X_day["__date"] = cur_dt
        X_list.append(X_day)

    X_all = pd.concat(X_list, ignore_index=True)

    X_model = align_rf_input(X_all.drop(columns=["__date"]), X_train_columns)
    delta_pred = model.predict(X_model)
    pred_qty = np.maximum(baseline_row["mean_qty"] + delta_pred, 0)

    curve_daily = X_all.copy()
    curve_daily["pred_qty"] = pred_qty

    curve_summary = (
        curve_daily
        .groupby(["discount_rate", "promo_type"], as_index=False)["pred_qty"]
        .sum()
        .rename(columns={"pred_qty": "cum_pred_qty"})
        .sort_values("discount_rate")
        .reset_index(drop=True)
    )

    return curve_summary, curve_daily



In [19]:
def recommend_discount_simple(curve_df, inventory_qty, expiry_days):
    """
    재고 수량과 유통기한을 기준으로 필요한 일판매량을 계산하고,
    수요 예측 곡선에서 최소 할인율로 목표 판매량을 충족하는 할인율을 추천
    """
    required_daily_sales = inventory_qty / max(expiry_days, 1)

    curve_sorted = curve_df.sort_values("discount_rate")

    for _, r in curve_sorted.iterrows():
        if r["pred_qty"] >= required_daily_sales:
            return {
                "recommended_discount": float(r["discount_rate"]),
                "promo_type": r["promo_type"],
                "required_daily_sales": required_daily_sales,
                "pred_daily_sales": float(r["pred_qty"]),
                "decision_reason": "min_discount_meets_required_sales"
            }

    # 아무 할인율도 못 맞추면 최대 할인
    last = curve_sorted.iloc[-1]
    return {
        "recommended_discount": float(last["discount_rate"]),
        "promo_type": last["promo_type"],
        "required_daily_sales": required_daily_sales,
        "pred_daily_sales": float(last["pred_qty"]),
        "decision_reason": "max_discount_used"
    }


In [ ]:
def recommend_promotion(
    model,
    store_code,
    item_code,
    inventory_qty,
    expiry_days,   # ✅ 유지: 이걸로 기간 생성
    date,
    sku_baseline,
    sm_baseline,
    X_train_columns
):
    """
    목적:
    - date ~ (date + expiry_days - 1) 기간 동안
    - 할인율별 누적 예상 판매량(cum_pred_qty)을 계산하고
    - 재고(inventory_qty)를 넘는 옵션 중 최소 할인율을 추천
    """

    # 0. 타입 정합성
    store_code = int(store_code)
    item_code = str(item_code)
    inventory_qty = float(inventory_qty)
    expiry_days = max(int(expiry_days), 1)

    # ✅ 내부 end_date 계산 (외부 입력은 받지 않음)
    start_dt = pd.to_datetime(date)
    end_dt = start_dt + pd.Timedelta(days=expiry_days - 1)

    # 1. baseline 선택 (SKU → fallback)
    baseline_row, source = get_baseline_row(
        store_code,
        item_code,
        sku_baseline,
        sm_baseline
    )

    if baseline_row is None:
        return {
            "status": "fail",
            "reason": "baseline_not_found"
        }

    # 2. 기간 누적 곡선 계산
    curve_summary, curve_daily = predict_cumulative_curve_rf(
        model=model,
        baseline_row=baseline_row,
        store_code=store_code,
        item_code=item_code,
        start_date=start_dt,
        end_date=end_dt,
        X_train_columns=X_train_columns
    )

    # 3. 추천 로직: 누적 판매량이 재고 이상인 최소 할인율
    selected_row = None
    for _, r in curve_summary.iterrows():
        if r["cum_pred_qty"] >= inventory_qty:
            selected_row = r
            break

    if selected_row is None:
        selected_row = curve_summary.iloc[-1]
        decision_reason = "max_discount_used_no_option_clears_inventory"
    else:
        decision_reason = "min_discount_clears_inventory_by_expiry_days"

    # 4. 결과 반환 (기존 필드 최대 유지)
    return {
        "status": "success",
        "baseline_source": source,
        "date": str(date),
        "expiry_days": expiry_days,

        "inventory_qty": inventory_qty,

        "recommended_discount": float(selected_row["discount_rate"]),
        "promo_type": int(selected_row["promo_type"]),

        # ✅ 기존 pred_daily_sales 대신 기간 누적 판매량으로 반환
        "pred_total_sales": float(selected_row["cum_pred_qty"]),
        "decision_reason": decision_reason,

        "curve_detail": curve_summary[
            ["discount_rate", "promo_type", "cum_pred_qty"]
        ].reset_index(drop=True),

        # (선택) 날짜별 상세
        "curve_daily_detail": curve_daily[
            ["__date", "discount_rate", "promo_type", "pred_qty"]
        ].rename(columns={"__date": "date"}).reset_index(drop=True)
    }


### 프로모션 추천 로직 실행

In [ ]:
PATH = r"randomforest_conservative_delta_artifacts2.pkl"
BASE_PATH = r"경희대3호점_예시데이터"

with open(PATH, "rb") as f:
    artifacts = pickle.load(f)

model = artifacts["rf_model_conservative"]
X_train_columns = artifacts["X_train_columns"]

store_sku_baseline_kh3 = pd.read_csv(
    f"{BASE_PATH}\\kh3_sku_baseline_str.csv",
    encoding="utf-8-sig",
    dtype={"item_code": str},
)

store_sm_baseline_kh3 = pd.read_csv(
    f"{BASE_PATH}\\kh3_sm_drop_baseline.csv",
    encoding="utf-8-sig",
    dtype={"상품소분류코드": int, "item_code": str},
)

c:\Users\user\Desktop\exp_app\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\user\Desktop\exp_app\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
result = recommend_promotion(
    model=model,
    store_code=10734,
    item_code="062458",
    inventory_qty=3,
    expiry_days=7,
    date="2025-01-01",
    sku_baseline=store_sku_baseline_kh3,
    sm_baseline=store_sm_baseline_kh3,
    X_train_columns=X_train_columns
)